# Ex1_localized_isotropic_source

A nonmultiplying four group slab. The source is isotropic in angle,
occupies only the chosen energy group, and is distributed across the
two central cells to preserve symmetry. `TOTAL_SOURCE` is its
spatially integrated strength.


In [ ]:
from pathlib import Path
import sys

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    # The Pyodide kernel does not inherit packages installed in the
    # environment that built the JupyterLite site.
    import piplite
    await piplite.install('ipywidgets==8.1.7')
    import ipywidgets as widgets

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

working_directory = Path.cwd().resolve()
candidates = (
    working_directory, *working_directory.parents,
    working_directory / 'Ex_RP_SN_discrete_ordinates',
    working_directory / 'RPTT_PoliTO_GitHub' / 'Ex_RP_SN_discrete_ordinates',
)
PROJECT_DIRECTORY = next(
    (path for path in candidates if (path / 'sn_1d.py').is_file()), None
)
if PROJECT_DIRECTORY is None:
    raise FileNotFoundError('Cannot find sn_1d.py.')

sys.path.insert(0, str(PROJECT_DIRECTORY))
from sn_1d import SNSettings, SNSolver, SlabMesh, SlabSNProblem

DATA_DIRECTORY = PROJECT_DIRECTORY / 'data'
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True})
print(f'Input: {DATA_DIRECTORY}')


In [ ]:
def plot_angular_and_scalar_flux(result, group=0, direction='all',
                                 normalize=True):
    # Select one group and either scalar, all angular, or one direction.
    if not 0 <= group < result.scalar_flux.shape[0]:
        raise ValueError('GROUP_TO_PLOT is out of range.')
    x = result.problem.mesh.cell_centers
    angular = result.angular_flux[group].copy()
    scalar = result.scalar_flux[group].copy()
    if normalize:
        scale = np.max(np.abs(scalar))
        angular /= scale
        scalar /= scale
        ylabel = 'flux / maximum scalar flux of the group'
    else:
        ylabel = 'flux'
    signed_mu = np.r_[result.directions, -result.directions]
    fig, ax = plt.subplots()
    if direction == 'all':
        for ordinate, mu in enumerate(signed_mu):
            ax.plot(x, angular[ordinate], lw=1, alpha=0.75,
                    label=fr'$\psi(\mu={mu:+.3f})$')
    elif direction != 'scalar':
        if not isinstance(direction, (int, np.integer)):
            raise ValueError("direction must be 'all', 'scalar', or an index.")
        if not 0 <= direction < len(signed_mu):
            raise ValueError('The direction index is out of range.')
        mu = signed_mu[direction]
        ax.plot(x, angular[direction], lw=2,
                label=fr'$\psi(\mu={mu:+.3f})$')
    ax.plot(x, scalar, 'k', lw=2.8,
            ls='--' if direction not in ('all', 'scalar') else '-',
            label=fr'$\phi_{{{group + 1}}}$')
    for interface in result.problem.mesh.region_edges[1:-1]:
        ax.axvline(interface, color='0.6', ls='--', lw=0.8)
    ax.set(xlabel='$z$ [cm]', ylabel=ylabel,
           title=f'Flux — group {group + 1}')
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    return fig, ax


def flux_widget(result, initial_group=0, initial_direction='all',
                normalize=True):
    # Interactive group/direction selection without rerunning the solver.
    groups = result.scalar_flux.shape[0]
    signed_mu = np.r_[result.directions, -result.directions]
    direction_options = [('All angular directions', 'all'),
                         ('Scalar flux only', 'scalar')]
    direction_options.extend(
        (f'Direction {index}: mu={mu:+.5f}', index)
        for index, mu in enumerate(signed_mu)
    )
    valid_directions = {value for _, value in direction_options}
    if not 0 <= initial_group < groups:
        raise ValueError(f'initial_group must be between 0 and {groups - 1}.')
    if initial_direction not in valid_directions:
        raise ValueError('initial_direction is not available.')

    group_control = widgets.Dropdown(
        options=[(f'Group {group + 1}', group) for group in range(groups)],
        value=initial_group, description='Energy group',
    )
    direction_control = widgets.Dropdown(
        options=direction_options, value=initial_direction,
        description='Direction',
        layout=widgets.Layout(width='310px'),
    )
    normalize_control = widgets.Checkbox(
        value=normalize, description='Normalize by max scalar flux'
    )

    def redraw(group, direction, normalize_flux):
        plot_angular_and_scalar_flux(
            result, group=group, direction=direction,
            normalize=normalize_flux,
        )
        plt.show()

    output = widgets.interactive_output(
        redraw,
        {
            'group': group_control,
            'direction': direction_control,
            'normalize_flux': normalize_control,
        },
    )
    controls = widgets.HBox(
        [group_control, direction_control, normalize_control]
    )
    return widgets.VBox([controls, output])


## Initial guess for inner iterations

Set `USE_PREVIOUS_FLUX_AS_INNER_GUESS` to `True` to reuse the current
angular flux. On the first outer iteration of a power step, this is the
angular flux from the preceding power step. Set it to `False` to restart
every group solve from unit angular flux. In the fixed source problem,
the same switch controls reuse between outer iterations.


In [ ]:
USE_PREVIOUS_FLUX_AS_INNER_GUESS = True
SOURCE_GROUP = 0       # Zero based group index: 0, ..., G-1
TOTAL_SOURCE = 1.0

settings = SNSettings(
    order=4, inner_tolerance=1e-6, outer_tolerance=1e-5,
    multigroup_warm_start=USE_PREVIOUS_FLUX_AS_INNER_GUESS,
)
solver = SNSolver.from_case(
    DATA_DIRECTORY, case='homogeneous', multigroup=True, settings=settings
)

mesh = solver.problem.mesh
slab_center = 0.5 * (mesh.region_edges[0] + mesh.region_edges[-1])
distance = np.abs(mesh.cell_centers - slab_center)
central_cells = np.isclose(distance, distance.min())
external_source = np.zeros((solver.problem.cross_sections.number_of_groups,
                            mesh.number_of_cells))
external_source[SOURCE_GROUP, central_cells] = (
    TOTAL_SOURCE / mesh.cell_widths[central_cells].sum()
)
result = solver.solve_fixed_source(external_source)

print(f'source cells = {np.flatnonzero(central_cells).tolist()}')
print(f'source cell centers = {mesh.cell_centers[central_cells]} cm')
print(f'spatial integral = '
      f'{np.sum(external_source[SOURCE_GROUP] * mesh.cell_widths):.6f}')
print(f'outer iterations = {result.outer_iterations}')


The source profile identifies its position and energy group.


In [ ]:
fig, ax = plt.subplots()
ax.step(mesh.cell_centers, external_source[SOURCE_GROUP], where='mid')
ax.set(xlabel='$z$ [cm]', ylabel='$Q_g(z)$',
       title=f'External source in group {SOURCE_GROUP + 1}');


## Fixed source scalar flux and outer iterations

The shaded interval marks the localized source, as in the fixed source
example in the original tutorial. The second panel shows inner solves
per outer iteration, summed over all energy groups. A fixed source
problem has no power iterations or $k_{eff}$.


In [ ]:
fig, (ax_flux, ax_iterations) = plt.subplots(1, 2, figsize=(12, 4.5))
for group, flux in enumerate(result.scalar_flux, start=1):
    ax_flux.plot(mesh.cell_centers, flux, label=f'Group {group}')
ax_flux.axvspan(mesh.cell_edges[np.flatnonzero(central_cells)[0]],
                mesh.cell_edges[np.flatnonzero(central_cells)[-1] + 1],
                color='tab:orange', alpha=0.2, label='External source')
ax_flux.set(xlabel='$z$ [cm]', ylabel='Scalar flux',
            title='Fixed source solution')
ax_flux.legend()

outer_counts = [sum(inner.iterations for inner in outer.inner_iterations)
                for outer in result.history]
ax_iterations.plot(range(1, len(outer_counts) + 1), outer_counts, 'o-')
ax_iterations.set(xlabel='Outer iteration',
                  ylabel='Inner iterations across all groups',
                  title='Fixed source iteration cost')
ax_iterations.set_xticks(range(1, len(outer_counts) + 1))
fig.tight_layout();


## Interactive angular and scalar flux

Select the energy group and one ordinate direction from the menus.
`All angular directions` overlays every angular flux, while
`Scalar flux only` hides them. With normalization enabled, angular and
scalar fluxes are divided by the maximum scalar flux of the selected
group, preserving their relative magnitudes.


In [ ]:
INITIAL_GROUP = 0       # Zero-based Python group index
INITIAL_DIRECTION = 'all'  # 'all', 'scalar', or a zero-based direction index
NORMALIZE_FLUX = True

flux_explorer = flux_widget(
    result,
    initial_group=INITIAL_GROUP,
    initial_direction=INITIAL_DIRECTION,
    normalize=NORMALIZE_FLUX,
)
display(flux_explorer)
